> ## SOLUTION / ANSWER KEY &mdash; Lab 10.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-fairness-across-groups.ipynb`](../lab-06-fairness-across-groups.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.6 &mdash; Fairness Across Groups

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Compute an outcome rate for each group
- Flag disparate impact with the 80% rule
- See why an average can hide unfairness

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; guardrail logic, tool wiring, agent structure, and reading a fixed real message trace &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Bias & fairness](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-10-06"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
An agent that **acts** can act on bias at scale (deck slide 5). You can't see unfairness in an **average**
&mdash; you must measure **per group**. A common test is the **80% rule**: if the lowest group's outcome
rate is less than 80% of the highest, that's **disparate impact** worth investigating. Make bias
**visible and measured**, not assumed away.

In [ ]:
# records: each is {group, approved}. We measure approval rate PER group.
print("example:", {"group": "A", "approved": True})

## Your Turn
Complete `approval_rate_by_group` and `disparate_impact` (the 80% rule).

In [ ]:
from collections import defaultdict

def approval_rate_by_group(records):
    tot, ok = defaultdict(int), defaultdict(int)
    for r in records:
        tot[r["group"]] += 1
        if r["approved"]:
            ok[r['group']] += 1
    return {g: ok[g] / tot[g] for g in tot}

def disparate_impact(rates, threshold=0.8):
    # the 80% rule: min rate / max rate below threshold flags disparate impact
    lo, hi = min(rates.values()), max(rates.values())
    return (lo / hi) < threshold

try:
    recs = ([{'group': 'A', 'approved': True}] * 8 + [{'group': 'A', 'approved': False}] * 2 +
            [{'group': 'B', 'approved': True}] * 5 + [{'group': 'B', 'approved': False}] * 5)
    rates = approval_rate_by_group(recs)
    print('rates:', rates)
    print('disparate impact?', disparate_impact(rates))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("computes a per-group rate", lambda: approval_rate_by_group([{"group": "A", "approved": True}, {"group": "A", "approved": False}])["A"] == 0.5)
expect_true("equal rates -> no disparate impact", lambda: disparate_impact({"A": 0.6, "B": 0.6}) is False)
expect_true("a large gap flags disparate impact", lambda: disparate_impact({"A": 0.8, "B": 0.5}) is True)
expect_true("a ratio above 0.8 is not flagged", lambda: disparate_impact({"A": 1.0, "B": 0.85}) is False)
expect_true("handles more than two groups", lambda: disparate_impact({"A": 0.9, "B": 0.9, "C": 0.5}) is True)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Measure per group, not on average -- an average of 65% can hide one group at 90% and another at 40%. The 80% rule makes disparate impact visible so a human can investigate. Machines aren't neutral; measure it.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>